# Customer Churn Prediction

Customer churn prediction is a critical problem for subscription-based businesses. 
Predicting which customers are likely to leave allows companies to proactively 
take retention actions.

This project builds a machine learning pipeline to predict whether a customer 
will churn using the Telco Customer Churn dataset.

The project includes:

• Exploratory Data Analysis (EDA)  
• Data preprocessing and feature engineering  
• Training multiple machine learning models  
• Model evaluation and comparison  
• Model interpretation using feature importance  
• Saving the final model for deployment

The goal is to identify the most important factors influencing customer churn 
and build a predictive model that can support retention strategies.

## 1️⃣ Load Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, confusion_matrix, classification_report, roc_curve

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

import pickle

sns.set(style='whitegrid')

## 2️⃣ Load Dataset

In [ ]:
# Load raw dataset
df = pd.read_csv('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.head()

## 3️⃣ Exploratory Data Analysis

In this section we analyze the Telco customer dataset to understand:
- **Target variable** (Churn) distribution and class balance
- **Business drivers** of churn: contract type, internet service, pricing, tenure
- **Relationships** between features and churn for model design and stakeholder insights

### Data Overview

Understand dataset size, column types, and data quality (missing values, outliers).

In [ ]:
# --- Data Overview: Shape, Dtypes, Missing Values ---
print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns\n")
print("Data types and non-null counts:")
df.info()
print("\nMissing values per column:")
df.isnull().sum()

# --- Dataset integrity checks (important for data quality) ---
# Duplicate rows can skew model training and metrics
duplicate_rows = df.duplicated().sum()
print(f"\nDuplicate rows: {duplicate_rows}")

# customerID should be unique per customer; duplicates indicate data issues
if 'customerID' in df.columns:
    dup_ids = df['customerID'].duplicated().sum()
    print("Duplicate customerIDs:", dup_ids)
    if dup_ids > 0:
        print("WARNING: Duplicate customer IDs found. Review data for quality issues.")

if duplicate_rows > 0:
    print("WARNING: Duplicate rows found. Consider dropping before modeling.")

In [ ]:
# --- Summary statistics for numeric features ---
df.describe()

### Churn Distribution

The target variable balance affects modeling: imbalanced data requires stratified sampling and metrics like F1 or ROC-AUC.

In [ ]:
# --- Churn Distribution ---
# Balanced vs imbalanced target affects model choice and evaluation metrics
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
sns.countplot(x='Churn', data=df, palette=['#2ecc71', '#e74c3c'], ax=ax[0])
ax[0].set_title('Churn Distribution (Count)', fontsize=12)
ax[0].set_ylabel('Number of Customers')
for p in ax[0].patches:
    ax[0].annotate(f'{p.get_height():.0f}', (p.get_x() + p.get_width()/2., p.get_height()),
                   ha='center', va='bottom', fontsize=11)

# Pie chart for proportion
churn_counts = df['Churn'].value_counts()
ax[1].pie(churn_counts, labels=churn_counts.index, autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'],
          explode=(0, 0.05), startangle=90)
ax[1].set_title('Churn Proportion', fontsize=12)
plt.tight_layout()
plt.show()

# Insight
print("Insight: Class imbalance —", f"{churn_counts['Yes']/len(df)*100:.1f}% churned. Consider stratified sampling and metrics like F1/ROC-AUC.")

### Churn vs Contract Type

Longer contracts typically show lower churn due to commitment and discounts. Month-to-month customers have more flexibility to leave.

In [ ]:
# --- Churn vs Contract Type ---
fig, ax = plt.subplots(figsize=(8, 5))
contract_order = ['Month-to-month', 'One year', 'Two year']
ct = pd.crosstab(df['Contract'], df['Churn']).reindex(contract_order)
ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
ct_pct.plot(kind='bar', stacked=True, ax=ax, color=['#2ecc71', '#e74c3c'], width=0.7)
ax.set_title('Churn Rate by Contract Type', fontsize=12)
ax.set_ylabel('Percentage')
ax.set_xlabel('Contract Type')
ax.legend(title='Churn', bbox_to_anchor=(1.02, 1))
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()
print("Insight: Month-to-month contracts show highest churn. One-year and two-year contracts retain customers better.")

### Churn vs Internet Service

Internet service type affects both pricing and customer satisfaction. Fiber optic may attract higher-paying but more demanding customers.

In [ ]:
# --- Churn vs Internet Service ---
fig, ax = plt.subplots(figsize=(8, 5))
ct = pd.crosstab(df['InternetService'], df['Churn']).reindex(['No', 'DSL', 'Fiber optic'])
ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
ct_pct.plot(kind='bar', stacked=True, ax=ax, color=['#2ecc71', '#e74c3c'], width=0.7)
ax.set_title('Churn Rate by Internet Service', fontsize=12)
ax.set_ylabel('Percentage')
ax.set_xlabel('Internet Service')
ax.legend(title='Churn', bbox_to_anchor=(1.02, 1))
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()
print("Insight: Fiber optic customers often show higher churn; DSL and no-internet segments differ. Key segment for retention focus.")

### Churn vs Monthly Charges

Higher monthly charges can indicate price sensitivity or premium plans. Compare distribution of charges for churned vs retained customers.

In [ ]:
# --- Churn vs Monthly Charges ---
fig, ax = plt.subplots(figsize=(8, 5))
sns.kdeplot(data=df, x='MonthlyCharges', hue='Churn', fill=True, palette=['#2ecc71', '#e74c3c'], alpha=0.6, ax=ax)
ax.set_title('Monthly Charges Distribution by Churn', fontsize=12)
ax.set_xlabel('Monthly Charges ($)')
ax.set_ylabel('Density')
plt.tight_layout()
plt.show()
print("Insight: Churned customers often have higher monthly charges (e.g., fiber + add-ons). Price/value perception drives churn.")

### Churn vs Tenure

Tenure (months as customer) is a strong predictor: new customers churn more; long-tenure customers tend to stay.

In [ ]:
# --- Churn vs Tenure ---
fig, ax = plt.subplots(figsize=(8, 5))
sns.kdeplot(data=df, x='tenure', hue='Churn', fill=True, palette=['#2ecc71', '#e74c3c'], alpha=0.6, ax=ax)
ax.set_title('Tenure Distribution by Churn', fontsize=12)
ax.set_xlabel('Tenure (months)')
ax.set_ylabel('Density')
plt.tight_layout()
plt.show()
print("Insight: Churned customers skew toward low tenure. Retention efforts in the first few months are critical.")

### Correlation Analysis (Numeric Features)

Correlations help identify redundant or highly related features and guide feature selection.

In [ ]:
# --- Correlation heatmap (numeric columns only) ---
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
plt.figure(figsize=(10, 6))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', center=0, fmt='.2f',
            linewidths=0.5, square=True)
plt.title('Correlation Matrix (Numeric Features)', fontsize=12)
plt.tight_layout()
plt.show()

## 4️⃣ Data Cleaning & Preprocessing

### Data Cleaning & Feature Engineering

This section prepares the raw Telco dataset for machine learning.

Main steps:
- Remove non-predictive identifiers
- Fix incorrect data types (TotalCharges)
- Encode target variable
- Encode categorical variables
- Scale numerical features (done after split to avoid leakage)
- Save cleaned dataset and preprocessing artifacts

In [ ]:
import os

# --- Config ---
RANDOM_STATE = 42
CARDINALITY_THRESHOLD = 30  # One-hot if unique <= this; else frequency-encode
save_splits = False  # Set True to also save train/test splits

os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../model', exist_ok=True)

# (a) Drop customerID safely
if 'customerID' in df.columns:
    df = df.drop(columns=['customerID'])
    print("Dropped customerID")
else:
    print("customerID not present; skipped")

# (b) Convert TotalCharges to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
n_coerced = df['TotalCharges'].isna().sum()
fill_val = df['TotalCharges'].median()
df['TotalCharges'] = df['TotalCharges'].fillna(fill_val)
print(f"Converted TotalCharges → {n_coerced} coerced to NaN; filled with median {fill_val:.2f}")

# (c) Encode target Churn → Churn_Yes (0/1)
df['Churn_Yes'] = (df['Churn'] == 'Yes').astype(int)
df = df.drop(columns=['Churn'])
assert set(df['Churn_Yes'].unique()).issubset({0, 1}), "Churn_Yes must be 0 or 1"
print("Encoded Churn → Churn_Yes (0/1)")

# (d) Detect categorical columns (pandas-version-safe)
cat_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()

# (e) Split into low vs high cardinality
cols_onehot = [c for c in cat_cols if df[c].nunique() <= CARDINALITY_THRESHOLD]
cols_freq = [c for c in cat_cols if df[c].nunique() > CARDINALITY_THRESHOLD]

# (f) One-hot encode low-cardinality columns
if cols_onehot:
    df = pd.get_dummies(df, columns=cols_onehot, drop_first=True)
    print(f"One-hot encoded: {cols_onehot} ({df.shape[1]} total features after encoding)")
else:
    print("No low-cardinality categorical columns to one-hot encode")

# Frequency-encode high-cardinality columns
if cols_freq:
    for col in cols_freq:
        freqs = df[col].value_counts(normalize=True)
        df[col] = df[col].map(freqs)
        df = df.rename(columns={col: f"{col}_freq"})
    print(f"Frequency-encoded columns: {cols_freq}")
else:
    print("No high-cardinality categorical columns found")

# (g)–(h) Save cleaned dataframe (scaling done after split to avoid leakage)
df.to_csv('../data/processed/cleaned_telco_churn.csv', index=False)
print("Saved cleaned dataframe to ../data/processed/cleaned_telco_churn.csv")

# (j) Optionally save train/test splits
if save_splits:
    X = df.drop(columns=['Churn_Yes'])
    y = df['Churn_Yes']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
    X_train.to_csv('../data/processed/X_train.csv', index=False)
    X_test.to_csv('../data/processed/X_test.csv', index=False)
    y_train.to_csv('../data/processed/y_train.csv', index=False)
    y_test.to_csv('../data/processed/y_test.csv', index=False)
    print("Saved splits: X_train, X_test, y_train, y_test → ../data/processed/")

# (k) Sanity checks and summary
obj_remain = df.select_dtypes(include=["object", "string"]).columns.tolist()
if obj_remain:
    print(f"Warning: object/string columns remain: {obj_remain}")
else:
    print("Sanity: no object dtype among features")
print(f"\nCleaned data shape: {df.shape[0]} rows × {df.shape[1]} columns")
print("\nCleaned dtypes:")
df.info()
df.head()

## 5️⃣ Split Data

In [ ]:
# Load processed dataset
df = pd.read_csv('../data/processed/cleaned_telco_churn.csv')

# Define features and target
X = df.drop("Churn_Yes", axis=1)
y = df["Churn_Yes"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale after split to avoid data leakage: fit on train, transform both
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])
with open('../model/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("Saved scaler to ../model/scaler.pkl")

print("Train set:", X_train.shape)
print("Test set: ", X_test.shape)

## 6️⃣ Train Models

In [ ]:
# --- Model Training and Evaluation ---

RANDOM_STATE = 42

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)
    results[name] = {'model': model, 'Accuracy': acc, 'ROC_AUC': roc_auc, 'y_pred': y_pred, 'y_prob': y_prob}

    print("=" * 60)
    print(f"  {name}")
    print("=" * 60)
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  ROC-AUC:  {roc_auc:.4f}")
    print("\n  Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\n  Classification Report:")
    print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))
    print()

# ROC curve comparison
plt.figure(figsize=(8, 6))
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    auc = res['ROC_AUC']
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], 'k--', label='Random baseline')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves for Churn Prediction Models')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

# Model comparison table
comparison = pd.DataFrame([
    {"Model": name, "Accuracy": r["Accuracy"], "ROC_AUC": r["ROC_AUC"]}
    for name, r in results.items()
]).sort_values("ROC_AUC", ascending=False)
print("Model comparison (sorted by ROC-AUC):")
print(comparison.to_string(index=False))
print()

# Select best model by ROC-AUC
best_name = max(results.keys(), key=lambda k: results[k]['ROC_AUC'])
best_model = results[best_name]['model']
best_roc = results[best_name]['ROC_AUC']

print("=" * 60)
print(f"  Best model: {best_name} (ROC-AUC = {best_roc:.4f})")
print("=" * 60)
print(f"  Saved to ../model/churn_model.pkl")

with open('../model/churn_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

# Save feature names to ensure same feature ordering during inference
with open('../model/feature_names.pkl', 'wb') as f:
    pickle.dump(X_train.columns.tolist(), f)

## 7️⃣ Feature Importance (Random Forest Example)

In [ ]:
# Feature names must match the training data (X_train)
feature_names = X_train.columns
print(f"Best model: {best_name}")

if best_name == "Random Forest":
    # Extract feature importances from the fitted model
    importances = best_model.feature_importances_
    feature_importance_df = pd.DataFrame({
        "Feature": feature_names,
        "Importance": importances
    }).sort_values("Importance", ascending=False)

    plt.figure(figsize=(10, 6))
    sns.barplot(x="Importance", y="Feature", data=feature_importance_df.head(20))
    plt.title("Top 20 Feature Importances (Random Forest)")
    plt.show()

elif best_name == "Logistic Regression":
    # Extract coefficients from the fitted model
    coefficients = best_model.coef_[0]
    feature_coef_df = pd.DataFrame({
        "Feature": feature_names,
        "Coefficient": coefficients
    }).sort_values("Coefficient", key=abs, ascending=False)

    plt.figure(figsize=(10, 6))
    sns.barplot(x="Coefficient", y="Feature", data=feature_coef_df.head(20))
    plt.title("Top 20 Feature Effects (Logistic Regression)")
    plt.show()

## 8️⃣ Model Saved

Best model (selected by ROC-AUC) is saved to `../model/churn_model.pkl` in the Training section above.

In [ ]:
# Best model already saved in Training section above
# Verify it exists:
import os
path = '../model/churn_model.pkl'
print(f"Best model saved at: {os.path.abspath(path)}" if os.path.exists(path) else "Run Training cell first.")